# 02 Data Cleaning – Uyazi
**Goal:** turn the raw 2009 H1N1 Flu Survey data into a clean, model-ready table, using what we learned in `01 Exploratory Data Analysis`.
**Input:** `../data/training_set_features.csv` + `../data/training_set_labels.csv`
**Output:** `../data/training_clean_uyazi.csv` (and `df_clean` in memory)

Cleaning decisions (each is explained in its own section):
1. Merge features and labels, set `respondent_id` as the index.
2. Tidy text values (stray whitespace, inconsistent labels).
3. Handle missing values: add *missingness flags* for the heavily missing columns, then fill.
4. Fix data types (binary, ordinal, categorical).
5. Validate the result with assertions.

> **Rule for this notebook:** only *constant* or *rule-based* fills are applied to the whole dataset. Anything learned from the data (mode, median) is flagged, because it should be re-fitted inside each CV fold in the modelling notebook to avoid data leakage.

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)

X = pd.read_csv("../data/training_set_features.csv")
y = pd.read_csv("../data/training_set_labels.csv")
raw = X.merge(y, on="respondent_id", validate="one_to_one")
TARGETS = ["h1n1_vaccine", "seasonal_vaccine"]

df = raw.copy()          # work on a copy; `raw` stays untouched for before/after comparisons
print(raw.shape)

## 1. Structure and identifiers

In [ ]:
assert df["respondent_id"].is_unique, "respondent_id should be unique"
df = df.set_index("respondent_id")
print("Duplicate answer rows:", df.duplicated().sum())
print("Shape:", df.shape)

## 2. Tidy text values
The EDA output showed `census_msa` contains a double space (`"MSA, Not Principle  City"`). We strip and collapse whitespace in every text column so categories are not split by invisible differences.

In [ ]:
cat_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()

for c in cat_cols:
    df[c] = (df[c].astype("string")
                  .str.strip()
                  .str.replace(r"\s+", " ", regex=True))

for c in ["census_msa", "income_poverty", "age_group", "education"]:
    print(c, "->", sorted(df[c].dropna().unique()))

## 3. Missing values
Missing share per column, before cleaning:

In [ ]:
missing = (df.isna().mean().mul(100).round(1)
             .sort_values(ascending=False))
missing[missing > 0].to_frame("% missing")

### 3a. Missingness flags (heavily missing columns)
Columns with more than **5 %** missing get an extra `<col>_missing` flag *before* filling. In surveys, a skipped question is often informative: for example `health_insurance` is ~46 % missing, and `employment_industry` / `employment_occupation` are missing mostly for people who are not employed. The flag lets a model use that signal.

In [ ]:
FLAG_THRESHOLD = 5  # percent
flag_cols = missing[missing > FLAG_THRESHOLD].index.tolist()
for c in flag_cols:
    df[f"{c}_missing"] = df[c].isna().astype("int8")
print("Flags added for:", flag_cols)

### 3b. Fill text (categorical) columns
Missing text answers become an explicit `"Missing"` category. This is a constant fill, so it cannot leak information.

For `employment_industry` and `employment_occupation`, missingness is expected when someone is not employed, so we use a more meaningful label for those rows.

In [ ]:
cat_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()

# Rule-based: not employed -> "Not employed", otherwise unknown -> "Missing"
not_employed = df["employment_status"].isin(["Not in Labor Force", "Unemployed"])
for c in ["employment_industry", "employment_occupation"]:
    df.loc[df[c].isna() & not_employed, c] = "Not employed"

for c in cat_cols:
    df[c] = df[c].fillna("Missing")

print("Remaining missing in text columns:", int(df[cat_cols].isna().sum().sum()))

### 3c. Fill numeric / binary / Likert columns
These are survey answers coded as numbers (`0/1` or `1–5`). Rule-based fills first:

- `doctor_recc_h1n1` and `doctor_recc_seasonal` are always missing together (same 2,160 rows), so they were probably not asked. They get the `_missing` flag from 3a.
- Remaining gaps are small (< 5 %); we fill with the **most frequent value (mode)**.

> ⚠️ The mode is learned from the data. This is a small, accepted simplification for the cleaning stage. In the modelling notebook, move this step into a `sklearn` `Pipeline` (`SimpleImputer(strategy="most_frequent")`) so it is fitted on training folds only.

In [ ]:
num_cols = [c for c in df.select_dtypes(include="number").columns
            if c not in TARGETS and not c.endswith("_missing")]

modes = df[num_cols].mode().iloc[0]
df[num_cols] = df[num_cols].fillna(modes)

print("Remaining missing values overall:", int(df.isna().sum().sum()))

## 4. Data types
- Binary and count columns → small integers.
- Ordered categories (`age_group`, `education`, `income_poverty`) → ordered `Categorical`, so they can be encoded ordinally later.
- Everything else → unordered `category`.

In [ ]:
binary_cols = [c for c in num_cols if set(df[c].unique()) <= {0.0, 1.0}]
likert_cols = [c for c in num_cols if c.startswith("opinion_") or c in ("h1n1_concern", "h1n1_knowledge")]
count_cols  = ["household_adults", "household_children"]

for c in binary_cols + likert_cols + count_cols:
    df[c] = df[c].astype("int8")

ordered = {
    "age_group": ["18 - 34 Years", "35 - 44 Years", "45 - 54 Years", "55 - 64 Years", "65+ Years"],
    "education": ["< 12 Years", "12 Years", "Some College", "College Graduate"],
    "income_poverty": ["Below Poverty", "<= $75,000, Above Poverty", "> $75,000"],
}
for c, order in ordered.items():
    # "Missing" is kept as an extra level (unordered here); the `<col>_missing` flag also records it.
    # Encode the real levels ordinally later and handle "Missing" via the flag.
    unexpected = set(df[c].unique()) - set(order) - {"Missing"}
    assert not unexpected, f"Unexpected labels in {c}: {unexpected}"
    df[c] = pd.Categorical(df[c], categories=order + ["Missing"], ordered=False)

nominal = [c for c in df.select_dtypes(include=["object", "string"]).columns]
for c in nominal:
    df[c] = df[c].astype("category")

df.dtypes.value_counts()

## 5. Validation

In [ ]:
# 1. no missing values left
assert df.isna().sum().sum() == 0

# 2. same rows as the raw data, targets untouched
assert len(df) == len(raw)
assert (df[TARGETS].values == raw.set_index("respondent_id")[TARGETS].values).all()

# 3. value ranges
assert df[binary_cols].isin([0, 1]).all().all()
assert df[[c for c in likert_cols if c.startswith("opinion_")]].isin([1, 2, 3, 4, 5]).all().all()
assert df["h1n1_concern"].between(0, 3).all() and df["h1n1_knowledge"].between(0, 2).all()
assert df[count_cols].between(0, 3).all().all()

print("All checks passed.")
print("Shape raw  :", raw.shape)
print("Shape clean:", df.shape, f"(+{df.shape[1] - (raw.shape[1] - 1)} missingness flags)")

Quick sanity check: cleaning must not change the vaccination rates.

In [ ]:
pd.DataFrame({
    "raw %": raw[TARGETS].mean().mul(100).round(1),
    "clean %": df[TARGETS].mean().mul(100).round(1),
})

## 6. Save

In [ ]:
df_clean = df.copy()
df_clean.to_csv("../data/training_clean_uyazi.csv")        # index = respondent_id
df_clean.head()

## Summary of cleaning decisions

| Issue | Decision | Why |
|---|---|---|
| Duplicate / non-unique ids | Checked, none found | Data is one row per respondent |
| Stray whitespace in labels | Stripped and collapsed | Avoids split categories (`census_msa`) |
| Columns > 5 % missing | `<col>_missing` flag added | Skipped questions can be informative (`health_insurance`, `doctor_recc_*`, `income_poverty`, `employment_*`) |
| Missing text answers | `"Missing"` category; `"Not employed"` for job columns when not working | Constant / rule-based, no leakage |
| Missing numeric answers | Mode fill (to be moved into the CV pipeline) | Small gaps; keeps scale and types |
| Types | Small ints, ordered levels, `category` | Memory and clearer encoding later |

**Next (modelling notebook):** one-hot / ordinal encode inside a `Pipeline`, stratified CV, ROC AUC for both targets.